# Лабораторная работа №1.2 — сегментация Vaihingen

**Цель:** сравнить FP32 и смешанную точность FP16/FP32 при обучении модели сегментации на наборе [Vaihingen Cropped](https://www.kaggle.com/datasets/naydex/vaihingen-cropped).

Измеряем время обучения, пиковую выделенную GPU-память, Accuracy, mean IoU и Categorical Crossentropy; отслеживаем потери и нестабильность. Весь код находится в редактируемых ячейках. **Среда:** Google Colab с GPU (T4 или лучше). Сначала выполните ячейки по порядку.

**Важно:** данные скачиваются с Kaggle; если потребуется авторизация, настройте доступ к Kaggle в Colab. Результаты будут рассчитаны после выполнения, а не заданы заранее.

## 1. Установка и настройки

In [ ]:
%pip -q install kagglehub

In [ ]:
import os, gc, time, random, copy, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
import kagglehub
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "Включите GPU: Среда выполнения → Сменить среду выполнения → GPU"
DEVICE = torch.device("cuda")
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
# Изменяйте параметры здесь. Для полного эксперимента увеличьте EPOCHS.
EPOCHS = 3
BATCH_SIZE = 8
BASE_CHANNELS = 24
LEARNING_RATE = 1e-3
MAX_TRAIN_SAMPLES = None  # например 1000 для быстрого теста; None = вся обучающая часть
NUM_WORKERS = 2
torch.backends.cudnn.benchmark = True

## 2. Загрузка набора данных

Загружаем датасет с Kaggle и исследуем имена и формы `.npy`. Для экономии RAM используем `np.load(..., mmap_mode='r')`, если массив не упакован в архив. Не переводим весь набор изображений в float32.

In [ ]:
DATA_ROOT = pathlib.Path(kagglehub.dataset_download("naydex/vaihingen-cropped"))
print("Данные:", DATA_ROOT)
npy_files = sorted(DATA_ROOT.rglob("*.npy"))
for p in npy_files:
    try:
        a = np.load(p, mmap_mode="r", allow_pickle=False)
        print(str(p.relative_to(DATA_ROOT)), "shape=", a.shape, "dtype=", a.dtype)
        del a
    except Exception as exc:
        print("Пропуск:", p, exc)
assert npy_files, "Не найдены файлы .npy; проверьте структуру загруженного датасета."

## 3. Подготовка изображений и масок

По умолчанию программа пытается определить массивы изображений и разметки по названиям и форме. Если в конкретной версии набора данных структура отличается, укажите пути вручную в `IMAGE_FILE` и `MASK_FILE`. Поддерживаются изображения `N,H,W,C` и `N,C,H,W`, маски с индексами классов, one-hot или RGB-палитрой.

In [ ]:
IMAGE_FILE = None  # например: DATA_ROOT / "images.npy"
MASK_FILE = None   # например: DATA_ROOT / "masks.npy"

def inspect(p):
    a = np.load(p, mmap_mode="r", allow_pickle=False)
    return a.shape, a.dtype

def is_image(shape):
    return len(shape) == 4 and (shape[-1] in (3, 4) or shape[1] in (3, 4))

def is_mask(shape):
    return len(shape) in (3, 4)

if IMAGE_FILE is None or MASK_FILE is None:
    meta = [(p, *inspect(p)) for p in npy_files]
    image_candidates = [(p, sh) for p, sh, dt in meta if is_image(sh) and any(t in p.name.lower() for t in ("image", "img", "rgb", "x_train", "train_x", "data"))]
    if not image_candidates:
        image_candidates = [(p, sh) for p, sh, dt in meta if is_image(sh)]
    if IMAGE_FILE is None:
        assert image_candidates, "Не удалось определить изображения. Задайте IMAGE_FILE вручную."
        IMAGE_FILE = image_candidates[0][0]
    image_shape, _ = inspect(IMAGE_FILE)
    mask_candidates = [(p, sh) for p, sh, dt in meta if p != IMAGE_FILE and is_mask(sh) and sh[0] == image_shape[0] and any(t in p.name.lower() for t in ("mask", "label", "gt", "y_train", "train_y", "target"))]
    if not mask_candidates:
        mask_candidates = [(p, sh) for p, sh, dt in meta if p != IMAGE_FILE and is_mask(sh) and sh[0] == image_shape[0]]
    if MASK_FILE is None:
        assert mask_candidates, "Не удалось определить маски. Задайте MASK_FILE вручную."
        MASK_FILE = mask_candidates[0][0]

images = np.load(IMAGE_FILE, mmap_mode="r", allow_pickle=False)
masks = np.load(MASK_FILE, mmap_mode="r", allow_pickle=False)
print("Изображения:", IMAGE_FILE, images.shape, images.dtype)
print("Маски:", MASK_FILE, masks.shape, masks.dtype)
assert len(images) == len(masks), "Число изображений и масок различается"
assert len(images) >= 2, "Нужно минимум два примера"
# Убедитесь, что это действительно соответствующие друг другу изображения и маски.
def image_to_hwc(a):
    if a.ndim == 3 and a.shape[0] in (3,4) and a.shape[-1] not in (3,4):
        return np.moveaxis(a, 0, -1)
    return a
def mask_to_hw(a):
    if a.ndim == 2: return a
    if a.ndim == 3 and a.shape[0] <= 20 and a.shape[-1] > 20:
        return np.argmax(a, axis=0) if a.shape[0] > 1 else a[0]
    if a.ndim == 3 and a.shape[-1] <= 20:
        if a.shape[-1] == 1: return a[...,0]
        if a.shape[-1] not in (3,4): return np.argmax(a, axis=-1)
        return a[...,:3]
    raise ValueError(f"Неожиданная форма маски: {a.shape}")
sample_img = image_to_hwc(np.asarray(images[0]))
sample_mask = mask_to_hw(np.asarray(masks[0]))
print("Один пример:", sample_img.shape, sample_mask.shape)
assert sample_img.shape[:2] == sample_mask.shape[:2], "Размеры изображения и маски не совпадают"

### Кодирование классов и визуализация

Для RGB-масок формируем таблицу уникальных цветов по всему набору. Для индексных масок используем уникальные значения. При большом числе различных цветов проверьте, что маски не были сжаты с потерями.

In [ ]:
from collections import Counter

def raw_mask(i):
    return mask_to_hw(np.asarray(masks[i]))

# Определяем палитру без загрузки всех масок одновременно.
rgb_masks = sample_mask.ndim == 3
values = set()
for i in range(len(masks)):
    m = raw_mask(i)
    if rgb_masks:
        packed = (m[...,0].astype(np.uint32)<<16) | (m[...,1].astype(np.uint32)<<8) | m[...,2].astype(np.uint32)
        values.update(np.unique(packed).tolist())
    else:
        values.update(np.unique(m).tolist())
    if len(values) > 256:
        raise ValueError("Слишком много уникальных значений маски (>256). Проверьте выбор MASK_FILE и формат разметки.")
CLASS_VALUES = np.array(sorted(values), dtype=np.int64)
NUM_CLASSES = len(CLASS_VALUES)
assert NUM_CLASSES >= 2, "Не найдено хотя бы два класса"
print("Число классов:", NUM_CLASSES, "значения:", CLASS_VALUES.tolist())

def encode_mask(a):
    if a.ndim == 3:
        a = (a[...,0].astype(np.uint32)<<16) | (a[...,1].astype(np.uint32)<<8) | a[...,2].astype(np.uint32)
    idx = np.searchsorted(CLASS_VALUES, a.astype(np.int64))
    return idx.astype(np.int64)

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for j, idx in enumerate(np.linspace(0, len(images)-1, 4, dtype=int)):
    x = image_to_hwc(np.asarray(images[idx]))
    y = encode_mask(raw_mask(idx))
    axes[0,j].imshow(x[...,:3]); axes[0,j].set_title(f"Изображение {idx}")
    axes[1,j].imshow(y, cmap="tab20", vmin=0, vmax=NUM_CLASSES-1); axes[1,j].set_title("Разметка")
    axes[0,j].axis("off"); axes[1,j].axis("off")
plt.tight_layout(); plt.show()

## 4. Dataset и разбиение train/validation

Делим на обучающую и валидационную выборки в соотношении 80/20 с фиксированным seed. Если кропы получены из одних и тех же исходных снимков, случайное разбиение может завысить оценку качества: для строгого эксперимента разделяйте по исходным снимкам.

In [ ]:
class VaihingenDataset(Dataset):
    def __len__(self): return len(images)
    def __getitem__(self, idx):
        x = image_to_hwc(np.asarray(images[idx]))
        y = encode_mask(raw_mask(idx))
        x = np.array(x[...,:3], copy=True)
        # uint8 храним до получения батча, затем конвертируем в float32
        return torch.from_numpy(x).permute(2,0,1), torch.from_numpy(np.array(y, copy=True))

dataset = VaihingenDataset()
perm = np.random.default_rng(SEED).permutation(len(dataset))
n_val = max(1, int(0.2 * len(dataset)))
train_idx, val_idx = perm[n_val:], perm[:n_val]
if MAX_TRAIN_SAMPLES is not None: train_idx = train_idx[:MAX_TRAIN_SAMPLES]
train_loader = DataLoader(Subset(dataset,train_idx.tolist()),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
val_loader = DataLoader(Subset(dataset,val_idx.tolist()),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
print("Train:",len(train_idx),"Validation:",len(val_idx))

## 5. U-Net для сегментации

Используем одну и ту же архитектуру и одинаковые начальные веса для обоих режимов.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, a, b):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                               nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)

class SmallUNet(nn.Module):
    def __init__(self, num_classes, b=24):
        super().__init__()
        self.e1=DoubleConv(3,b); self.e2=DoubleConv(b,2*b); self.e3=DoubleConv(2*b,4*b)
        self.pool=nn.MaxPool2d(2)
        self.center=DoubleConv(4*b,8*b)
        self.u3=DoubleConv(12*b,4*b); self.u2=DoubleConv(6*b,2*b); self.u1=DoubleConv(3*b,b)
        self.out=nn.Conv2d(b,num_classes,1)
    def forward(self,x):
        import torch.nn.functional as F
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.e3(self.pool(b))
        z=self.center(self.pool(c))
        for skip,layer in ((c,self.u3),(b,self.u2),(a,self.u1)):
            z=F.interpolate(z,size=skip.shape[-2:],mode="bilinear",align_corners=False)
            z=layer(torch.cat([z,skip],dim=1))
        return self.out(z)

base=SmallUNet(NUM_CLASSES,BASE_CHANNELS)
initial=copy.deepcopy(base.state_dict())
print("Параметров:",sum(p.numel() for p in base.parameters()))
del base

## 6. Метрики и обучение FP32 / Mixed Precision

Accuracy — доля верно классифицированных пикселей. Mean IoU — среднее пересечение/объединение по классам, встречающимся в разметке или предсказании. Categorical Crossentropy вычисляется через `torch.nn.functional.cross_entropy` по логитам и индексам классов (эквивалент для one-hot целей). Стабильность оцениваем по NaN/Inf и динамике потерь.

In [ ]:
import torch.nn.functional as F

@torch.inference_mode()
def validate(model, mixed):
    model.eval()
    cm = torch.zeros((NUM_CLASSES,NUM_CLASSES),device=DEVICE,dtype=torch.int64)
    total_loss=0.0; pixels=0
    for x,y in val_loader:
        x=x.to(DEVICE,non_blocking=True).float()/255.0
        y=y.to(DEVICE,non_blocking=True).long()
        with torch.autocast("cuda",dtype=torch.float16,enabled=mixed):
            logits=model(x)
            loss=F.cross_entropy(logits.float(),y,reduction="sum")
        pred=logits.argmax(1)
        cm += torch.bincount((y*NUM_CLASSES+pred).reshape(-1),minlength=NUM_CLASSES**2).reshape(NUM_CLASSES,NUM_CLASSES)
        total_loss+=loss.item();pixels+=y.numel()
    diag=cm.diag().float(); union=cm.sum(0)+cm.sum(1)-diag
    iou=(diag/union.clamp(min=1))[union>0]
    return {"Accuracy":(diag.sum()/cm.sum().clamp(min=1)).item(),
            "mIoU":iou.mean().item() if len(iou) else float("nan"),
            "Crossentropy":total_loss/max(1,pixels)},cm.cpu()

def train_experiment(mode):
    mixed = mode == "Mixed FP16/FP32"
    gc.collect(); torch.cuda.empty_cache()
    model=SmallUNet(NUM_CLASSES,BASE_CHANNELS).to(DEVICE)
    model.load_state_dict(initial)
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    scaler=torch.amp.GradScaler("cuda",enabled=mixed)
    history=[]; bad_steps=0
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0=time.perf_counter()
    for epoch in range(EPOCHS):
        model.train(); loss_sum=0.0; batches=0
        for x,y in train_loader:
            x=x.to(DEVICE,non_blocking=True).float()/255.0
            y=y.to(DEVICE,non_blocking=True).long()
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda",dtype=torch.float16,enabled=mixed):
                logits=model(x)
                loss=F.cross_entropy(logits.float(),y)
            if not torch.isfinite(loss):
                bad_steps+=1; continue
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            loss_sum+=loss.item();batches+=1
        metrics,cm=validate(model,mixed)
        row={"epoch":epoch+1,"train_loss":loss_sum/max(1,batches),**metrics}
        history.append(row)
        print(f"{mode} | эпоха {epoch+1}/{EPOCHS} | loss={row['train_loss']:.4f} | acc={metrics['Accuracy']:.4f} | mIoU={metrics['mIoU']:.4f}")
    torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0  # включает валидацию после каждой эпохи
    peak=torch.cuda.max_memory_allocated()/1024**2
    weights=sum(p.numel()*p.element_size() for p in model.parameters())/1024**2
    summary={"Режим":mode,"Время train+val, с":elapsed,"Accuracy":history[-1]["Accuracy"],
             "mIoU":history[-1]["mIoU"],"Categorical Crossentropy":history[-1]["Crossentropy"],
             "Пик GPU памяти, МиБ":peak,"Веса, МиБ":weights,"NaN/Inf шаги":bad_steps}
    state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    del model,optimizer,scaler
    gc.collect();torch.cuda.empty_cache()
    return summary,pd.DataFrame(history),state

results=[]; histories={}; trained_states={}
for mode in ("FP32","Mixed FP16/FP32"):
    summary,h,state=train_experiment(mode)
    results.append(summary);histories[mode]=h;trained_states[mode]=state

## 7. Сравнение режимов и графики

In [ ]:
comparison=pd.DataFrame(results)
display(comparison.round(4))
speedup=comparison.iloc[0]["Время train+val, с"]/comparison.iloc[1]["Время train+val, с"]
memory_ratio=comparison.iloc[0]["Пик GPU памяти, МиБ"]/comparison.iloc[1]["Пик GPU памяти, МиБ"]
print(f"Отношение времени FP32 / Mixed: {speedup:.2f}x (больше 1 = ускорение)")
print(f"Отношение пиковой памяти FP32 / Mixed: {memory_ratio:.2f}x")
fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax,metric in zip(axes,("train_loss","Accuracy","mIoU")):
    for name,h in histories.items(): ax.plot(h["epoch"],h[metric],marker="o",label=name)
    ax.set_title(metric);ax.set_xlabel("Эпоха");ax.grid(True);ax.legend()
plt.tight_layout();plt.show()
comparison.to_csv("vaihingen_fp32_vs_mixed.csv",index=False)
print("Результаты сохранены: vaihingen_fp32_vs_mixed.csv")

## 8. Визуализация предсказаний

Показываем исходное изображение, правильную маску и прогноз обученной Mixed Precision модели.

In [ ]:
model=SmallUNet(NUM_CLASSES,BASE_CHANNELS).to(DEVICE)
model.load_state_dict(trained_states["Mixed FP16/FP32"])
model.eval()
fig,axes=plt.subplots(3,3,figsize=(12,11))
for j,idx in enumerate(val_idx[:3]):
    x,y=dataset[int(idx)]
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
        pred=model((x.unsqueeze(0).to(DEVICE).float()/255.0)).argmax(1)[0].cpu().numpy()
    axes[0,j].imshow(x.permute(1,2,0).numpy());axes[0,j].set_title(f"Изображение {idx}")
    axes[1,j].imshow(y.numpy(),cmap="tab20",vmin=0,vmax=NUM_CLASSES-1);axes[1,j].set_title("Истинная маска")
    axes[2,j].imshow(pred,cmap="tab20",vmin=0,vmax=NUM_CLASSES-1);axes[2,j].set_title("Предсказание")
    for row in axes: row[j].axis("off")
plt.tight_layout();plt.show()
del model;torch.cuda.empty_cache()

## 9. Вывод

После выполнения сравните время, память, Accuracy, mIoU, Crossentropy и кривые потерь. Ускорение FP16 не гарантируется на любой GPU; пик `max_memory_allocated()` отражает только память, выделенную PyTorch. Оба режима сохраняют параметры модели в FP32; Mixed Precision использует FP16 в подходящих операциях и GradScaler для градиентов. Если оценка на случайном разбиении оказывается завышенной, повторите опыт с разбиением по исходным ортофотопланам.